In [0]:
CATALOGO = "workspace"
ORIGEM = f"{CATALOGO}.bronze.pncp_contratacoes"
DESTINO = f"{CATALOGO}.silver.contratacoes"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.silver")

from pyspark.sql import functions as F, types as T, Window

bronze = spark.table(ORIGEM)

ESQUEMAS_ANINHADOS = {
    "orgaoEntidade": "cnpj STRING, esferaId STRING, poderId STRING, razaoSocial STRING",
    "unidadeOrgao": ("codigoIbge STRING, codigoUnidade STRING, municipioNome STRING, "
                     "nomeUnidade STRING, ufNome STRING, ufSigla STRING"),
    "amparoLegal": "codigo INT, descricao STRING, nome STRING",
}

tipos_bronze = {campo.name: campo.dataType for campo in bronze.schema.fields}

conversoes = [
    F.from_json(F.col(coluna), esquema).alias(coluna)
    for coluna, esquema in ESQUEMAS_ANINHADOS.items()
    if isinstance(tipos_bronze.get(coluna), T.StringType)
]

if conversoes:
    bronze = bronze.withColumns(conversoes)

def ts(coluna: str):
    return F.to_timestamp(F.col(coluna).cast("string"))


def valor(coluna: str):
    return F.col(coluna).cast(T.DecimalType(18, 2))

In [0]:
selecionado = bronze.select(
    F.col("numeroControlePNCP").alias("numero_controle_pncp"),
    F.col("numeroCompra").alias("numero_compra"),
    F.col("anoCompra").cast("int").alias("ano_compra"),
    F.col("sequencialCompra").cast("int").alias("sequencial_compra"),
    F.col("processo").alias("processo"),
    F.col("orgaoEntidade.cnpj").alias("orgao_cnpj"),
    F.trim(F.col("orgaoEntidade.razaoSocial")).alias("orgao_razao_social"),
    F.col("orgaoEntidade.esferaId").alias("orgao_esfera_id"),
    F.col("orgaoEntidade.poderId").alias("orgao_poder_id"),
    F.col("unidadeOrgao.codigoUnidade").alias("unidade_codigo"),
    F.trim(F.col("unidadeOrgao.nomeUnidade")).alias("unidade_nome"),
    F.col("unidadeOrgao.codigoIbge").alias("municipio_codigo_ibge"),
    F.trim(F.col("unidadeOrgao.municipioNome")).alias("municipio_nome"),
    F.col("unidadeOrgao.ufSigla").alias("uf_sigla"),
    F.col("modalidadeId").cast("int").alias("modalidade_id"),
    F.col("modalidadeNome").alias("modalidade_nome"),
    F.col("modoDisputaId").cast("int").alias("modo_disputa_id"),
    F.col("modoDisputaNome").alias("modo_disputa_nome"),
    F.col("situacaoCompraId").cast("int").alias("situacao_compra_id"),
    F.col("situacaoCompraNome").alias("situacao_compra_nome"),
    F.col("tipoInstrumentoConvocatorioNome").alias("tipo_instrumento_nome"),
    F.col("amparoLegal.nome").alias("amparo_legal_nome"),
    F.col("srp").cast("boolean").alias("srp"),
    F.regexp_replace(F.trim(F.col("objetoCompra")), r"\s+", " ").alias("objeto_compra"),
    ts("dataPublicacaoPncp").alias("data_publicacao_pncp"),
    ts("dataInclusao").alias("data_inclusao"),
    ts("dataAberturaProposta").alias("data_abertura_proposta"),
    ts("dataEncerramentoProposta").alias("data_encerramento_proposta"),
    ts("dataAtualizacaoGlobal").alias("data_atualizacao_global"),
    valor("valorTotalEstimado").alias("valor_total_estimado"),
    valor("valorTotalHomologado").alias("valor_total_homologado"),
    F.col("_arquivo_origem").alias("arquivo_origem"),
    F.col("_data_ingestao").alias("data_ingestao"),
    F.col("_execucao_id").alias("execucao_id"),
)

In [0]:
# Deduplicação: mantém a versão mais recente de cada contratação.
# Necessária porque janelas de datas e execuções repetidas podem trazer o mesmo registro.
janela = Window.partitionBy("numero_controle_pncp").orderBy(
    F.col("data_atualizacao_global").desc_nulls_last(),
    F.col("data_ingestao").desc_nulls_last(),
)

deduplicado = (
    selecionado
    .withColumn("_rn", F.row_number().over(janela))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)


In [0]:
silver = (
    deduplicado
    .withColumn("ano_mes_publicacao", F.date_format("data_publicacao_pncp", "yyyy-MM"))
    .withColumn("data_publicacao", F.to_date("data_publicacao_pncp"))
    .withColumn("estimado_valido", F.col("valor_total_estimado") > 0)
    .withColumn("tem_homologado", F.col("valor_total_homologado") > 0)
    .withColumn(
        "prazo_proposta_dias",
        F.datediff("data_encerramento_proposta", "data_abertura_proposta"),
    )
    .withColumn("prazo_proposta_invalido", F.col("prazo_proposta_dias") < 0)

    .withColumn(
        "desfecho",
        F.when(F.col("situacao_compra_nome").isin("Revogada", "Anulada"), F.lit("frustrada"))
         .when(F.col("situacao_compra_nome") == "Suspensa", F.lit("suspensa"))
         .when(F.col("valor_total_homologado") > 0, F.lit("homologada"))
         .otherwise(F.lit("sem_homologacao")),
    )

    .withColumn(
        "economia_valor",
        F.when(
            F.col("estimado_valido") & F.col("tem_homologado"),
            F.col("valor_total_estimado") - F.col("valor_total_homologado"),
        ),
    )
    .withColumn(
        "razao_homologado_estimado",
        F.when(
            F.col("estimado_valido") & F.col("tem_homologado"),
            F.round(F.col("valor_total_homologado") / F.col("valor_total_estimado"), 4)
             .cast(T.DecimalType(12, 4)),
        ),
    )
    .withColumn(
        "economia_pct",
        F.round((1 - F.col("razao_homologado_estimado")).cast("double") * 100, 1),
    )
)


(
    silver.write.mode("overwrite")
    .option("overwriteSchema", "true")
    .clusterBy("modalidade_id", "municipio_codigo_ibge")
    .saveAsTable(DESTINO)
)

In [0]:
spark.sql(f"""
COMMENT ON TABLE {DESTINO} IS
'Camada Silver: contratações do PNCP (UF=PA) limpas e padronizadas. Uma linha por numero_controle_pncp. \
Campos aninhados achatados, datas em timestamp, valores em decimal(18,2). Problemas de qualidade não são \
excluídos: ficam marcados nas colunas estimado_valido, tem_homologado, razao_suspeita, \
prazo_proposta_invalido e apta_analise_economia. Origem: bronze.pncp_contratacoes.'
""")

display(spark.sql(f"""
SELECT
  COUNT(*)                                        AS linhas,
  COUNT(DISTINCT numero_controle_pncp)            AS contratacoes_distintas,
  SUM(CASE WHEN NOT estimado_valido THEN 1 ELSE 0 END)        AS estimado_zerado,
  SUM(CASE WHEN NOT tem_homologado  THEN 1 ELSE 0 END)        AS sem_homologado,
  SUM(CASE WHEN prazo_proposta_invalido THEN 1 ELSE 0 END)    AS prazo_invalido
FROM {DESTINO}
"""))

display(spark.sql(f"""
SELECT modalidade_nome, desfecho, COUNT(*) AS qtd
FROM {DESTINO}
GROUP BY modalidade_nome, desfecho
ORDER BY modalidade_nome, qtd DESC
"""))

display(spark.table(DESTINO).limit(25))

linhas,contratacoes_distintas,estimado_zerado,sem_homologado,prazo_invalido
17432,17432,1002,99,0


modalidade_nome,desfecho,qtd
Concorrência - Eletrônica,homologada,804
Concorrência - Eletrônica,sem_homologacao,637
Concorrência - Eletrônica,frustrada,27
Concorrência - Eletrônica,suspensa,19
Concorrência - Presencial,sem_homologacao,18
Concorrência - Presencial,homologada,11
Concurso,sem_homologacao,5
Credenciamento,sem_homologacao,209
Credenciamento,homologada,41
Credenciamento,frustrada,10


numero_controle_pncp,numero_compra,ano_compra,sequencial_compra,processo,orgao_cnpj,orgao_razao_social,orgao_esfera_id,orgao_poder_id,unidade_codigo,unidade_nome,municipio_codigo_ibge,municipio_nome,uf_sigla,modalidade_id,modalidade_nome,modo_disputa_id,modo_disputa_nome,situacao_compra_id,situacao_compra_nome,tipo_instrumento_nome,amparo_legal_nome,srp,objeto_compra,data_publicacao_pncp,data_inclusao,data_abertura_proposta,data_encerramento_proposta,data_atualizacao_global,valor_total_estimado,valor_total_homologado,arquivo_origem,data_ingestao,execucao_id,ano_mes_publicacao,data_publicacao,estimado_valido,tem_homologado,prazo_proposta_dias,prazo_proposta_invalido,desfecho,economia_valor,razao_homologado_estimado,economia_pct
01611858000155-1-000013/2026,002,2026,13,021/2026,01611858000155,MUNICIPIO DE PLACAS,M,N,8035,FUNDO MUNICIPAL DE ASSISTENCIA SOCIAL,1505650,Placas,PA,6,Pregão - Eletrônico,1,Aberto,1,Divulgada no PNCP,Edital,"Lei 14.133/2021, Art. 28, I",false,"[Portal de Compras Públicas] - FORNECIMENTO DE CESTAS BÁSICAS E KITS DE HIGIENE PESSOAL, DEVIDAMENTE MONTADOS, EMBALADOS E IDENTIFICADOS, DESTINADOS AO ATENDIMENTO DE FAMÍLIAS EM SITUAÇÃO DE VULNERABILIDADE SOCIAL ACOMPANHADAS PELA SECRETARIA MUNICIPAL DE ASSISTÊNCIA SOCIAL DO MUNICÍPIO DE PLACAS–PA.",2026-03-02T11:56:16.000Z,2026-03-02T11:56:16.000Z,2026-03-02T23:59:00.000Z,2026-03-12T23:59:00.000Z,2026-04-17T11:46:40.000Z,1446216.00,1375440.00,dbfs:/Volumes/workspace/bronze/raw_pncp/contratacoes/execucao=20260920T005732/mod06_202603.jsonl,2026-09-20T01:35:43.901Z,20260920T005732,2026-03,2026-03-02,true,true,10,false,homologada,70776.00,0.9511,4.9
01611858000155-1-000022/2026,PE 4,2026,22,34/2026,01611858000155,MUNICIPIO DE PLACAS,M,N,980060,PREFEITURA MUNICIPAL DE PLACAS - PA,1505650,Placas,PA,6,Pregão - Eletrônico,1,Aberto,1,Divulgada no PNCP,Edital,"Lei 14.133/2021, Art. 28, I",true,"contratação de empresa(s) especializada(s) no fornecimento de água mineral sem gás, em diferentes apresentações, e gás de cozinha (GLP – botijão de 13 kg), destinados ao atendimento das demandas administrativas, institucionais e operacionais dos órgãos da Administração Direta e dos Fundos Municipais vinculados ao Poder Executivo Municipal",2026-04-15T15:53:55.000Z,2026-04-15T15:53:55.000Z,2026-04-15T17:00:00.000Z,2026-05-08T00:00:00.000Z,2026-05-28T17:05:34.000Z,753202.16,742068.37,dbfs:/Volumes/workspace/bronze/raw_pncp/contratacoes/execucao=20260920T005732/mod06_202604.jsonl,2026-09-20T01:35:43.901Z,20260920T005732,2026-04,2026-04-15,true,true,23,false,homologada,11133.79,0.9852,1.5
01611858000155-1-000030/2026,007,2026,30,054/2026,01611858000155,MUNICIPIO DE PLACAS,M,N,8008,FUNDO MUNICIPAL DE EDUCAÇÃO,1505650,Placas,PA,6,Pregão - Eletrônico,6,Fechado-Aberto,1,Divulgada no PNCP,Edital,"Lei 14.133/2021, Art. 28, I",false,"[Portal de Compras Públicas] - Contratação de empresa especializada para prestação de serviços comuns de transporte escolar terrestre, de natureza contínua, destinados ao atendimento dos alunos regularmente matriculados na rede pública municipal de ensino, compreendendo rotas da zona rural e urbana do Município.",2026-05-27T19:45:28.000Z,2026-05-27T19:45:28.000Z,2026-05-28T00:00:00.000Z,2026-06-11T00:00:00.000Z,2026-07-07T03:13:37.000Z,33662226.71,27561327.61,dbfs:/Volumes/workspace/bronze/raw_pncp/contratacoes/execucao=20260920T005732/mod06_202605.jsonl,2026-09-20T01:35:43.901Z,20260920T005732,2026-05,2026-05-27,true,true,14,false,homologada,6100899.10,0.8188,18.1
01611858000155-1-000039/2026,10,2026,39,71/2026,01611858000155,MUNICIPIO DE PLACAS,M,N,6805,Prefeitura Municipal de Placas,1505650,Placas,PA,6,Pregão - Eletrônico,6,Fechado-Aberto,1,Divulgada no PNCP,Edital,"Lei 14.133/2021, Art. 28, I",true,"[Portal de Compras Públicas] - REGISTRO DE PREÇOS PARA FUTURA E EVENTUAL CONTRATAÇÃO DE EMPRESA ESPECIALIZADA NO FORNECIMENTO PARCELADO E SOB DEMANDA DE MATERIAIS DE LIMPEZA, HIGIENIZAÇÃO E CONSERVAÇÃO, DESTINADOS AO ATENDIMENTO DAS NECESSIDADES ADMINISTRATIVAS E